In [1]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_validate
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, auc, accuracy_score)
from sklearn.preprocessing import label_binarize
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime

# === CONFIGURACIÓN ===
INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected"
BASE_OUTPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\models"
MODEL_NAME = "Logistic"
INTENTO = 4  # Cambiar manualmente

fecha_actual = datetime.today().strftime('%Y-%m-%d')
OUTPUT_PATH = os.path.join(BASE_OUTPUT_PATH, f"{MODEL_NAME}_{INTENTO:02d}_{fecha_actual}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

metricas_totales = []
mejor_modelo = None
mejor_score = -1

variantes_X = sorted([f for f in os.listdir(INPUT_PATH) if f.startswith("X_train")])

for x_file in variantes_X:
    base_name = x_file.replace("X_train_", "").replace(".parquet", "")
    try:
        print(f"\n🔍 Procesando: {base_name}")

        # Cargar datos
        X_train = pd.read_parquet(os.path.join(INPUT_PATH, f"X_train_{base_name}.parquet"))
        X_test = pd.read_parquet(os.path.join(INPUT_PATH, f"X_test_{base_name}.parquet"))
        y_train = pd.read_parquet(os.path.join(INPUT_PATH, f"y_train_{base_name}.parquet")).squeeze()
        y_test = pd.read_parquet(os.path.join(INPUT_PATH, f"y_test_{base_name}.parquet")).squeeze()

        model = LogisticRegression(max_iter=500, multi_class='multinomial', solver='lbfgs', class_weight='balanced')

        # Validación cruzada
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_scores = cross_validate(model, X_train, y_train, cv=cv,
                                   scoring=['accuracy', 'f1_weighted', 'f1_macro'],
                                   return_train_score=True)

        # Entrenamiento final
        start = time.time()
        model.fit(X_train, y_train)
        tiempo = (time.time() - start) / 60

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        report_dict = classification_report(y_test, y_pred, output_dict=True)
        df_report = pd.DataFrame(report_dict).transpose()
        df_report["tiempo_min"] = tiempo

        f1_1 = df_report.loc['1', 'f1-score'] if '1' in df_report.index else 0
        recall_1 = df_report.loc['1', 'recall'] if '1' in df_report.index else 0

        if f1_1 > mejor_score:
            mejor_score = f1_1
            mejor_modelo = base_name

        cm = confusion_matrix(y_test, y_pred)
        df_report.to_csv(os.path.join(OUTPUT_PATH, f"metricas_{base_name}.csv"))

        # PDF con visualizaciones
        with PdfPages(os.path.join(OUTPUT_PATH, f"reporte_{base_name}.pdf")) as pdf:
            ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap='Blues')
            plt.title("Matriz de Confusión")
            pdf.savefig()
            plt.close()

            # Curvas ROC por clase
            classes = np.unique(y_test)
            y_bin = label_binarize(y_test, classes=classes)
            plt.figure()
            for i, clase in enumerate(classes):
                fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
                roc_auc = auc(fpr, tpr)
                plt.plot(fpr, tpr, label=f"Clase {clase} (AUC={roc_auc:.2f})")
            plt.plot([0, 1], [0, 1], 'k--')
            plt.title("Curvas ROC")
            plt.xlabel("FPR")
            plt.ylabel("TPR")
            plt.legend()
            pdf.savefig()
            plt.close()

            # Resultados de validación cruzada
            plt.figure()
            for score_name in ["train_f1_weighted", "test_f1_weighted"]:
                plt.plot(cv_scores[score_name], marker='o', label=score_name)
            plt.title("F1 Weighted - Validación Cruzada")
            plt.xlabel("Fold")
            plt.ylabel("F1")
            plt.legend()
            pdf.savefig()
            plt.close()

        metricas_totales.append({
            "modelo": base_name,
            "f1_nivel_1": f1_1,
            "recall_nivel_1": recall_1,
            "tiempo_min": tiempo,
            "accuracy": accuracy_score(y_test, y_pred),
            "macro_f1_test": df_report.loc['macro avg', 'f1-score'],
            "weighted_f1_test": df_report.loc['weighted avg', 'f1-score'],
            "cv_f1_train_mean": np.mean(cv_scores['train_f1_weighted']),
            "cv_f1_test_mean": np.mean(cv_scores['test_f1_weighted']),
            "cv_macro_f1_test": np.mean(cv_scores['test_f1_macro']),
            "cv_accuracy": np.mean(cv_scores['test_accuracy'])
        })

        print(f"✅ {base_name}: F1_clase_1={f1_1:.3f}, Recall_clase_1={recall_1:.3f}, Tiempo={tiempo:.2f}min")

    except Exception as e:
        print(f"❌ Error en {base_name}: {e}")

# Guardar resumen total
pd.DataFrame(metricas_totales).to_csv(os.path.join(OUTPUT_PATH, "resumen_modelos_logistic_regression.csv"), index=False)
print(f"\n🏆 Mejor modelo según F1 para clase 1: {mejor_modelo} (F1 = {mejor_score:.4f})")



🔍 Procesando: MaxAbs_FE_ALL


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_l

✅ MaxAbs_FE_ALL: F1_clase_1=0.272, Recall_clase_1=0.323, Tiempo=9.81min

🔍 Procesando: MaxAbs_FE_ANOVA


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_l

✅ MaxAbs_FE_ANOVA: F1_clase_1=0.236, Recall_clase_1=0.271, Tiempo=2.71min

🔍 Procesando: MaxAbs_FE_MI


...

✅ Standard_ORIGINAL_ANOVA: F1_clase_1=0.237, Recall_clase_1=0.278, Tiempo=0.15min

🔍 Procesando: Standard_ORIGINAL_MI


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\U

✅ Standard_ORIGINAL_MI: F1_clase_1=0.261, Recall_clase_1=0.342, Tiempo=0.28min

🔍 Procesando: Standard_ORIGINAL_VAR


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\U

✅ Standard_ORIGINAL_VAR: F1_clase_1=0.270, Recall_clase_1=0.321, Tiempo=0.67min

🏆 Mejor modelo según F1 para clase 1: Standard_FE_ALL (F1 = 0.2786)
